In [58]:
#from mpl_toolkits import mplot3d
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split
from IPython.core.pylabtools import figsize
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from scipy import stats
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from scipy.stats import pearsonr,spearmanr
from itertools import combinations
from scipy.special import comb, perm
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import ShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

scaler_M = MinMaxScaler()

data=pandas.read_csv(r"..select by pearson from initial-MA-clipped - ranked.csv")

sf=np.array(data.sf)/100
Vs=np.array(data.sv_Volume)
Vso=np.array(data.so_Volume)
N_sv=np.array(data.N_sv)
N_so=np.array(data.N_so)

r2_list=[]

In [59]:
n_alphas = 100
alphas = np.logspace(-6, 0.6, n_alphas)

kr=GridSearchCV(KernelRidge(kernel='laplacian'), cv=10,
                 param_grid={"alpha": alphas,
                            "gamma": np.logspace(-10, 2, 50)})

In [70]:
F=np.array(data)
p=data.ev_ratio
#
kfold=2
n=82
#
p_train=np.array(p[0:n*kfold,])
F_train=F[0:n*kfold,3:67]

p_test=np.array(p[n*(kfold+1)-n:n*(kfold+1),])
F_test=F[n*(kfold+1)-n:n*(kfold+1),3:67]

(164,)

In [64]:
scaler_M.fit(F_train)
F_tr=scaler_M.transform(F_train)
    
resul= kr.fit(F_tr,(p_train.reshape(-1,1)-1)*100)
    
_alpha=resul.best_params_.pop('alpha')
_gamma=resul.best_params_.pop('gamma')

krr=KernelRidge(alpha=_alpha, 
                #kernel='rbf',
                kernel='laplacian',
                gamma=_gamma)

In [65]:
F_test=scaler_M.transform(F_test)

krr.fit(F_tr,(p_train.reshape(-1,1)-1)*100)
    
p_tr = krr.predict(F_tr)
r_2_tr=r2_score((p_train.reshape(-1,1)-1)*100, p_tr)
    
p_pre = krr.predict(F_test)   
r_2=r2_score((p_test.reshape(-1,1)-1)*100, p_pre)

print(r_2_tr)
print(r_2)

0.9057446052272982
-37.28667840227069


In [66]:
ind=[]
r2=[]

In [67]:
sub=63
for l in range(0,sub):
    
    evaluate_r_2=[]
    evaluate_r_2_tr=[]
    
    for k in range(0,np.shape(F_train)[1]):
        
        F_d=np.delete(F_train,k,1)
        F_dtest=np.delete(F_test,k,1)
            
        scaler_M.fit(F_d) 
        
        F_tr=scaler_M.transform(F_d)
        F_te=scaler_M.transform(F_dtest)
        
        krr.fit(F_tr,(p_train.reshape(-1,1)-1)*100)
            
        p_tr = krr.predict(F_tr)
        r_2_tr=r2_score((p_train.reshape(-1,1)-1)*100, p_tr)
        evaluate_r_2_tr.append(r_2_tr)
    
        p_pre = krr.predict(F_te)   
        r_2=r2_score((p_test.reshape(-1,1)-1)*100, p_pre)
        evaluate_r_2.append(r_2)
        
    r2.append(max(evaluate_r_2))    
    t=evaluate_r_2.index(max(evaluate_r_2))
    ind.append(t)
    
    F_train=np.delete(F_train,t,1)
    F_test=np.delete(F_test,t,1)

In [68]:
r2_list.append(max(r2)) 

In [19]:
r=pd.DataFrame(r2_list)
r.to_csv('r2_list20fold.csv')